# nano Beatrix — the thought phase on the framed Castor

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_thought_castor.ipynb)

The framed Castor (nano-castor-v1.2: the Castor body pretrained with half of every phase's rows framed; its end state on the Hub) continues through one phase of a billion bytes of the plain continuation stream under the same framed diet, now WITH THE THOUGHT KIND: the art thought dataset's documents rendered in the frame grammar (a thinking frame of typed child frames, the spec frame of records, the wordings, a question), one row in ten. Its control is the same phase from the same start under the diet without thoughts. Both ship to the nano training repo (experiments, not releases).

| | nano-castor-v1.2-th (the thought arm) | nano-castor-v1.2-tc (the control) |
|---|---|---|
| start | nano-castor-v1.2 at its end (step 61,422; weights and both optimizers) | the same |
| phase | thought_continuation, 1.0 B bytes of the plain continuation stream | the same |
| rows | half framed: carry .30, frames .20, caption .20, note .10, THOUGHT .20 (`framed_diet.FORMS['p50t']`) | half framed: carry .35, frames .25, caption .25, note .15 (`p50`) |
| ships to | AbstractPhil/alephllm-nano-beatrix-training | the same |

THE THOUGHT READS (thought_screen.py; the dataset plan's section 8) run on the source before the phase and on both arms at its end, on rows the loader holds out of training by series: the RECORD read (the value stands in the spec frame), the CHAIN read (a value the thoughts derive) in the trained layout A against the frameless layout C, the UNKNOWN read (the body answers unknown), the TRAVEL read (the same questions in layout D, sealed from training), the PRIOR beside them, and the same reads on trained rows (recall against generalisation). The marks, written before the run (plans/2026-10-20_thought_castor_run_plan.md): RECORD at or above 0.80 and 0.20 over the prior; CHAIN: A over C by 0.10 and over the prior by 0.10; UNKNOWN answered at or above 0.50 with false unknowns at most 0.10; TRAVEL: D within 0.10 of A; TEXT KEPT within 0.02 of the control; FRAMES KEPT within 0.10 of the control; RECALL: trained rows over held rows by at most 0.20.

Length: about 20 minutes a craft on an RTX PRO 6000 at the framed Castor's pace (3.35 steps a second), plus the frames corpus (built once), the gauges and the reads. The run resumes from the Hub when a session ends: rerun C1 to C3.


In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART RUNTIME if the version line below changes.
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.69"
import torch
import geolip.nano_beatrix as N
import geolip.alephllm as A
from geolip.nano_beatrix import nano_v1 as V1
from geolip.nano_beatrix import framed_diet as FDT
from geolip.nano_beatrix import thought_frames as TF
print('geolip.nano_beatrix', N.__version__, '| geolip.alephllm', A.__version__, '| torch', torch.__version__,
      '| card', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 69), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart, then rerun from C1')
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()                                   # the value is never printed
assert HF_TOKEN, 'HF_TOKEN missing: the caption pack reads its sources and the run ships to the repo'
os.environ['HF_TOKEN'] = HF_TOKEN

# ============ THE DECISION BLOCK — every setting from the record, with its source beside it ============
OUT = '/content/nano_thought'                         # the session folder: the births, the packs, the frames corpus, the runs, the bed's gauge folder
NAMES = ['nano-castor-v1.2-th', 'nano-castor-v1.2-tc']   # the thought arm and its control; the Pollux pair: 'nano-pollux-dual-v1.2-th', 'nano-pollux-dual-v1.2-tc'
REPO = V1.repo_of(NAMES[0])                           # AbstractPhil/alephllm-nano-beatrix-training: the experiments repo (Phil 2026-10-20), each craft under its own name
SOURCE = V1.body_of(NAMES[0])                         # nano-castor-v1.2: the start, read from V1.REPO (the release repo) at its end
SOURCE_STEP = V1.start_step(NAMES[0], V1.TOKENS_PER_STEP)
SHAPES = [(16, 4), (32, 2)]                           # rows a micro-batch x micro-batches a step, each 262,144 bytes; C2 benches them and keeps 16 x 4 unless another is 10% faster
GAUGE_EVERY = V1.GAUGE_EVERY                          # the bed's gauges every 4,000 steps and at every boundary
PACK_ROWS = None                                      # the caption pack: None = the first shards of each source whole; a number = that many rows a source (a smoke)
MAX_HOURS = None                                      # this session's budget in hours (None: until the phase ends or the cell is stopped)
PACK = f'{OUT}/caption_pack'
THOUGHT = f'{OUT}/thought_pack'                       # the art thought dataset's shards (AbstractPhil/thought-symbolic-image-captions), fetched once
print('decision block:', ', '.join(f"{n} ({V1.ALL[n]['arm']} seed {V1.ALL[n]['seed']}, framed {V1.framed_of(n)}, {V1.arm_text(n)})" for n in NAMES),
      '| start', SOURCE, 'at', SOURCE_STEP, 'from', V1.REPO, '| the readout on AdamW', V1.train_of(NAMES[0]),
      '| kinds', {n: dict(FDT.FORMS[V1.framed_of(n)]['kinds']) for n in NAMES}, '| shapes', SHAPES, '| out', OUT, '| ships to', REPO)


In [ ]:
# C2 (P) — PREFLIGHT: the births (the bodies' starts, checked), the held-out head, THE CAPTION PACK, THE THOUGHT PACK (the dataset's shards
# from the Hub: the rows by depth, the held series), THE BASELINE THOUGHT READS on the source before the phase, THE FRAMED STREAM with the
# thought kind on this machine (its pace, the kinds' shares, one decoded thought row), the 2s step at each shape, the push probe. Nothing trains here.
import json, time
import numpy as np
from huggingface_hub import HfApi
from geolip.nano_beatrix.bed_corpus import write_heldout
from geolip.nano_beatrix import caption_pack as CP
from geolip.nano_beatrix import frames_data as FD
from geolip.alephllm.data import special_tokens as ST
from geolip.alephllm.data.streams import build_stream
from geolip.alephllm.data.tokenizers import build_tokenizer
births = {n: V1.write_birth(V1.body_of(n), f'{OUT}/births') for n in NAMES}
held = write_heldout(f'{OUT}/bed')
print('births:', ', '.join(f'{n} <- {V1.body_of(n)} ({V1.ALL[n]["arm"]} seed {V1.ALL[n]["seed"]})' for n in NAMES), '| held-out head', os.path.basename(held))
stats = CP.ensure(PACK, HF_TOKEN, PACK_ROWS, log=print)
os.environ[CP.PACK_ENV] = PACK
PK, HELDP = CP.load(PACK), CP.load(PACK, 'held')
print(f'the caption pack: {PK.n:,} training documents; {HELDP.n:,} held')
TSTATS = TF.ensure(THOUGHT, HF_TOKEN, log=print)
os.environ[TF.PACK_ENV] = THOUGHT
TPK, THELD = TF.load(THOUGHT), TF.load(THOUGHT, 'held')
print(f'the thought pack: {TPK.n:,} training rows, {THELD.n:,} held rows (one series in ten; never trained)')

# THE BASELINE: the thought reads on the source at its end, before any thought row (the weights from the release repo)
t0 = time.time()
BASE = V1.thought_stored(SOURCE, SOURCE_STEP, OUT, repo=V1.REPO, token=HF_TOKEN, pack=THOUGHT)
BASE_H = dict(BASE['headline'], step=SOURCE_STEP)
print(f'baseline ({SOURCE} at {SOURCE_STEP:,}, {time.time() - t0:.0f} s, {BASE["n"]} held items):', {k: (round(v, 3) if isinstance(v, float) else v) for k, v in BASE['headline'].items()})
for ex in BASE['examples'][:8]:
    print('  ', ex)

# the framed stream as the run builds it: the fineweb-edu stream at micro-batch 1 under the wrapper with the thought kind
tok = build_tokenizer('byte-trigram')
inner = build_stream('fineweb-edu', tok, V1.CTX, 1, seed=1337)
fs = FDT.FramedStream(inner, 16, 1337, FDT.FORMS[V1.framed_of(NAMES[0])], pack=PK, thought=TPK)
NAMES_ST = {v: k for k, v in vars(ST).items() if isinstance(v, int) and 0xF0 <= v <= 0xFF and k.isupper()}
def show(ids):
    return ''.join(f'<{NAMES_ST[c]}>' if c in NAMES_ST else chr(c) if 32 <= c < 127 or c == 10 else f'\\x{c:02x}' for c in ids)
seen, n = {}, 0
first = fs.next_batch()                               # the stream's start outside the clock
t0 = time.time()
while time.time() - t0 < 15:
    ids, kind = fs.row()
    n += 1
    if kind not in seen:
        seen[kind] = ids
rate = n / (time.time() - t0)
need = 877_000 / FD.ROW_BYTES                         # rows a second the framed Castor's pace needs (877 kB/s on the G4)
print(f'the framed stream composes {rate:,.0f} rows a second against {need:.0f} needed at the framed Castor\'s pace'
      + ('' if rate > 1.5 * need else ' — TOO SLOW: the card would wait on the data'))
print('kinds over', n, 'rows:', FDT.share_table(fs.counts))
if 'thought' in seen:
    print('\n===== one thought row (the first 1,200 bytes) =====')
    print(show(seen['thought'][:1200].tolist()))

card_gb = torch.cuda.get_device_properties(0).total_memory / 2 ** 30
BENCH = []
for nm in NAMES[:1]:
    for mb, ga in SHAPES:
        r = V1.bench(nm, births[nm], mb, ga)
        BENCH.append(r)
        print(f'  {nm:<26} {mb:>2} x {ga}: {r["tok_s"] / 1e3:6.0f}k bytes a second, {r["step_s"]:.3f} s a step, peak {r["peak_gb"]:.1f} GB')
(MB, GA), why = V1.pick_shape(BENCH, card_gb)
print(f'-> the crafts train at {MB} x {GA}: {why}')
steps = V1.plan_steps(NAMES[0], MB * GA * V1.CTX) - V1.start_step(NAMES[0], MB * GA * V1.CTX)
secs = [r['step_s'] * steps for r in BENCH if (r['micro_batch'], r['grad_accum']) == (MB, GA)]
print(f'-> each phase trains about {secs[0] / 60:.0f} min on this card at the bench pace ({steps:,} steps), plus the frames corpus (built once), the gauges and the reads')

# the push probe (the ship-complete law): a write to the experiments repo and a read-back BEFORE any training.
api = HfApi(token=HF_TOKEN)
path = 'preflight/push_probe_thought.json'
body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'card': torch.cuda.get_device_name(0), 'bench': BENCH,
                   'shape': [MB, GA], 'crafts': {n: V1.ALL[n] for n in NAMES}, 'compose_rows_s': rate, 'kinds': fs.counts,
                   'caption_pack': [PK.n, HELDP.n], 'thought_pack': TSTATS, 'baseline': BASE_H}).encode()
api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message='nano thought phase: push probe')
PUSH_OK = path in set(api.list_repo_files(REPO))
print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path, '->', REPO)


In [ ]:
# C3 (T) — THE TRAIN: both arms through the thought phase from the source's end state, resuming from their records in the experiments repo;
# the trainer's own lines, the bed's gauge lines, the caption reads and THE THOUGHT READS ([v1]) print as they come; the boundary returns to
# the driver, which evals, gauges, reads, ships the boundary table and TensorBoard. Rerun the cell to continue (a new session too).
HIST = V1.run(OUT, HF_TOKEN if PUSH_OK else None, NAMES, micro_batch=MB, grad_accum=GA, repo=REPO, gauge_every=GAUGE_EVERY,
              max_hours=MAX_HOURS, caption_pack=PACK, caption_pack_rows=PACK_ROWS, thought_pack=THOUGHT)
from IPython.display import Markdown, display
display(Markdown(V1.thought_table({n: HIST.get(n) or [] for n in NAMES}, before=BASE_H, before_name=SOURCE)))
rows_th, rows_tc = HIST.get(NAMES[0]) or [], HIST.get(NAMES[1]) or [] if len(NAMES) > 1 else []
display(Markdown(V1.thought_marks_table(V1.thought_marks(rows_th[-1] if rows_th else None, rows_tc[-1] if rows_tc else None, BASE_H))))


In [ ]:
# C4 — THE TABLE AND THE MARKS: the boundary rows from the experiments repo (all sessions), the baseline, the marks by the pre-registered
# rules, and the curves. Run any time; it trains nothing (a new session reruns C1 and the first part of C2 for BASE_H).
import json
from IPython.display import Markdown, display
from huggingface_hub import hf_hub_download, snapshot_download
hub = snapshot_download(REPO, token=HF_TOKEN, local_dir=f'{OUT}/hub',
                        allow_patterns=[f'{n}/runs/*' for n in NAMES] + [f'{n}/v1_boundaries.json' for n in NAMES])
ROWS = {}
for n in NAMES:
    p = f'{hub}/{n}/v1_boundaries.json'
    ROWS[n] = json.load(open(p, encoding='utf-8')) if os.path.exists(p) else []
    display(Markdown(f'### {n}: {len(ROWS[n])} boundaries'))
display(Markdown(V1.thought_table(ROWS, before=BASE_H, before_name=SOURCE)))
rows_th, rows_tc = ROWS.get(NAMES[0]) or [], (ROWS.get(NAMES[1]) or []) if len(NAMES) > 1 else []
display(Markdown(V1.thought_marks_table(V1.thought_marks(rows_th[-1] if rows_th else None, rows_tc[-1] if rows_tc else None, BASE_H))))
%load_ext tensorboard
%tensorboard --logdir {hub}


### Changing the run

Edit the decision block in C1 (the pair: `NAMES = ['nano-pollux-dual-v1.2-th', 'nano-pollux-dual-v1.2-tc']` for the framed Pollux; the session's hours; the shapes), rerun C1 (the install is cached), then C2 and C3. A craft already in the experiments repo resumes from its last checkpoint; it never restarts. The thought pack is the dataset at the time C2 fetched it: a bigger dataset at a later run means a new session folder.
